# Module 09: TurboQuant Stage 2 -- Unbiased Inner Products

*Estimated time: ~30 minutes*

**Prerequisites**: You have built the MSE quantizer (Module 06) and QJL (Module 08).
Now we combine them into the full TurboQuant two-stage system.

This is the paper's key contribution: near-optimal distortion AND unbiased inner products.

**What you will learn**:
1. Why MSE-only quantization produces biased inner product estimates
2. The elegant (b-1) + 1 bit split that fixes the bias
3. How to build and verify a two-stage TurboQuant system
4. The bias-variance tradeoff between biased and unbiased modes
5. Why unbiased inner products matter for search and attention

In [ ]:
import sys; sys.path.insert(0, "..")
from src.helpers import check_intuition, check_close, section_header

import math
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

---
## 1. The problem with MSE-only quantization

The MSE quantizer from Module 06 minimizes reconstruction error. It does a great job at
that -- the reconstructed vector $\hat{x}$ is close to the original $x$ in Euclidean distance.

But there is a subtle problem: **inner product estimates are biased**.

What does that mean? If you quantize $x$ to $\hat{x}$ and compute $\langle y, \hat{x} \rangle$
as an estimate of $\langle y, x \rangle$, the expected value of your estimate is NOT equal
to the true value:

$$\mathbb{E}[\langle y, \hat{x} \rangle] \neq \langle y, x \rangle$$

The MSE quantizer systematically *shrinks* the reconstruction toward zero (a common property
of quantizers). This means inner products are systematically underestimated. Let's see this.

In [ ]:
section_header("The Bias Problem", "MSE quantization shrinks inner products")

We will build a toy MSE quantizer that captures the essential behavior: normalize, rotate,
scalar-quantize each coordinate, then inverse-rotate and rescale. This mimics what the
real TurboQuantMSE does.

In [ ]:
class ToyTurboQuantMSE:
    """Simplified MSE quantizer for demonstration.
    
    Applies a random rotation, then uniform scalar quantization to each
    coordinate, then inverse rotation. Stores norms separately.
    """
    def __init__(self, dim, bits=2, seed=0):
        self.dim = dim
        self.bits = bits
        self.n_levels = 2 ** bits
        # Random orthogonal rotation (simulates randomized Hadamard)
        gen = torch.Generator().manual_seed(seed)
        Q, _ = torch.linalg.qr(torch.randn(dim, dim, generator=gen))
        self.rotation = Q
        # Lloyd-Max centroids for standard normal, precomputed for small bit widths
        # These are the MSE-optimal centroids for N(0, 1/d) after rotation
        self._setup_quantizer()
    
    def _setup_quantizer(self):
        """Set up Lloyd-Max-style boundaries and centroids for N(0,1)."""
        # For simplicity, use uniform quantizer on a clipped range.
        # This is less optimal than Lloyd-Max but shows the bias clearly.
        clip = 3.0  # clip at +/- 3 sigma
        step = 2 * clip / self.n_levels
        self.boundaries = torch.linspace(-clip, clip, self.n_levels + 1)
        self.centroids = (self.boundaries[:-1] + self.boundaries[1:]) / 2
    
    def quantize(self, x):
        """Quantize: store norm, normalize, rotate, scalar quantize."""
        norms = x.norm(dim=-1, keepdim=True)
        x_unit = x / norms.clamp(min=1e-10)
        # Rotate
        x_rot = x_unit @ self.rotation.T
        # Scale to match expected coordinate magnitude after rotation
        # For unit vectors in d dims, each rotated coord ~ N(0, 1/d)
        scale = math.sqrt(self.dim)
        x_scaled = x_rot * scale
        # Quantize each coordinate
        codes = torch.bucketize(x_scaled, self.boundaries[1:-1])
        return {'codes': codes, 'norms': norms, 'scale': scale}
    
    def dequantize(self, quant_out):
        """Dequantize: look up centroids, inverse rotate, rescale by norm."""
        codes = quant_out['codes']
        norms = quant_out['norms']
        scale = quant_out['scale']
        # Look up centroids
        x_scaled = self.centroids[codes]
        x_rot = x_scaled / scale
        # Inverse rotate
        x_unit = x_rot @ self.rotation
        # Rescale
        return x_unit * norms


print("ToyTurboQuantMSE defined.")

Now let's demonstrate the bias. We create many random vectors, quantize them, compute
inner products with a query, and compare against the true inner products.

In [ ]:
torch.manual_seed(42)
dim = 64
n_vectors = 500

# Create random vectors
X = torch.randn(n_vectors, dim)
X = X / X.norm(dim=-1, keepdim=True)  # unit vectors

# Create a query
query = torch.randn(dim)
query = query / query.norm()

# True inner products
true_ip = X @ query

# MSE-only quantization (2 bits per coordinate)
mse_quantizer = ToyTurboQuantMSE(dim, bits=2, seed=0)
mse_out = mse_quantizer.quantize(X)
X_hat = mse_quantizer.dequantize(mse_out)

# Estimated inner products using quantized vectors
est_ip_mse = X_hat @ query

# Compute bias
bias = (est_ip_mse - true_ip).mean().item()
slope = (est_ip_mse * true_ip).sum() / (true_ip * true_ip).sum()  # linear regression slope

print(f"Mean bias (est - true): {bias:.4f}")
print(f"Regression slope (should be 1.0 for unbiased): {slope.item():.4f}")
print(f"\nThe slope < 1 means inner products are systematically SHRUNK.")

In [ ]:
# Plot: true IP vs estimated IP
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=true_ip.numpy(), y=est_ip_mse.numpy(),
    mode="markers",
    marker=dict(size=4, color="steelblue", opacity=0.5),
    name="MSE-only estimates",
))

# Perfect line (slope=1)
ip_range = [true_ip.min().item(), true_ip.max().item()]
fig.add_trace(go.Scatter(
    x=ip_range, y=ip_range,
    mode="lines",
    line=dict(color="green", width=2, dash="dash"),
    name="Perfect (slope=1)",
))

# Regression line
fig.add_trace(go.Scatter(
    x=ip_range, y=[slope.item() * v for v in ip_range],
    mode="lines",
    line=dict(color="red", width=2),
    name=f"Actual (slope={slope.item():.3f})",
))

fig.update_layout(
    title="MSE-Only Quantization: Inner Products Are Biased",
    xaxis_title="True inner product <y, x>",
    yaxis_title="Estimated inner product <y, x_hat>",
    template="plotly_white",
    height=500, width=600,
    legend=dict(x=0.02, y=0.98),
)

fig.add_annotation(
    x=0.5, y=0.1, xref="paper", yref="paper",
    text=f"Slope = {slope.item():.3f} (should be 1.0)<br>Systematic underestimation!",
    showarrow=False,
    font=dict(size=13, color="red"),
    bgcolor="rgba(255,255,255,0.8)",
)

fig.show()

The red line has slope less than 1. This means MSE-only quantization **systematically
underestimates** inner products. Large true inner products get estimated as smaller than
they actually are.

For search tasks (find the most similar vector), this bias can cause **rank inversions**:
a vector with a slightly smaller true inner product might get a higher estimated inner
product than the correct nearest neighbor, simply because the bias affects different
vectors differently.

---
## 2. The elegant solution: (b-1) + 1 bit split

The paper's insight is simple and beautiful:

- Give **(b-1) bits** to the MSE quantizer: this handles the bulk of the signal
- Give **1 bit** to QJL on the **residual**: this corrects the bias
- **Total: b bits** per coordinate, and the inner product estimate is unbiased

### Why the residual?

After MSE quantization, the residual $r = x - \hat{x}$ is small (because the MSE quantizer
is good at its job). QJL's 1-bit estimate is noisier for larger vectors, so operating on
the small residual keeps variance low.

The inner product decomposes cleanly:

$$\langle y, x \rangle = \langle y, \hat{x} \rangle + \langle y, r \rangle$$

The first term comes from the MSE reconstruction (biased but low variance). The second
term comes from QJL on the residual (unbiased by construction). Together:

$$\mathbb{E}[\langle y, \hat{x} \rangle + \widehat{\langle y, r \rangle}_{\text{QJL}}]
= \langle y, \hat{x} \rangle + \langle y, r \rangle
= \langle y, x \rangle$$

Unbiased! The MSE term is deterministic (no randomness once quantized), and QJL is unbiased
by its own construction.

---
## 3. Build the two-stage composition

We also need a toy QJL. This mirrors the reference QJL: project with a random Gaussian
matrix, store sign bits and norms.

In [ ]:
class SimpleQJL:
    """Simplified QJL for demonstration.
    
    Projects vectors with a random Gaussian matrix, stores sign bits
    and norms. Supports asymmetric inner product estimation.
    """
    def __init__(self, dim, proj_dim=None, seed=42):
        self.dim = dim
        self.proj_dim = proj_dim or dim
        gen = torch.Generator().manual_seed(seed)
        self.R = torch.randn(self.proj_dim, dim, generator=gen)
    
    def quantize(self, x):
        """Project and store sign bits + norms."""
        norms = x.norm(dim=-1)
        z = x @ self.R.T  # (..., proj_dim)
        sign_bits = (z >= 0).to(torch.uint8)
        return {'sign_bits': sign_bits, 'norms': norms}
    
    def estimate_ip(self, query, qjl_out):
        """Asymmetric inner product estimation.
        
        Full-precision query projection dotted with sign bits,
        scaled by stored norms and sqrt(pi/2)/m.
        """
        query_proj = query @ self.R.T  # (..., proj_dim)
        signs = qjl_out['sign_bits'].float() * 2 - 1  # {0,1} -> {-1,+1}
        norms = qjl_out['norms']
        # Raw dot product
        if query_proj.dim() == 1:
            raw = signs @ query_proj / self.proj_dim  # (n,)
        else:
            raw = query_proj @ signs.T / self.proj_dim  # (n_queries, n)
        # Scale by norms and sqrt(pi/2)
        return raw * norms * math.sqrt(math.pi / 2)


print("SimpleQJL defined.")

In [ ]:
class ToyTurboQuant:
    """Two-stage TurboQuant: MSE quantizer + QJL on residual.
    
    Allocates (bit_width - 1) bits to MSE and 1 bit to QJL.
    Produces unbiased inner product estimates.
    """
    def __init__(self, dim, bit_width=3, seed=0):
        self.dim = dim
        self.bit_width = bit_width
        self.mse = ToyTurboQuantMSE(dim, bits=bit_width - 1, seed=seed)
        self.qjl = SimpleQJL(dim, proj_dim=dim, seed=seed + 1)
    
    def quantize(self, x):
        """Stage 1: MSE quantize. Stage 2: QJL on residual."""
        mse_out = self.mse.quantize(x)
        x_hat = self.mse.dequantize(mse_out)
        residual = x - x_hat
        qjl_out = self.qjl.quantize(residual)
        return mse_out, qjl_out
    
    def compute_inner_product(self, query, mse_out, qjl_out):
        """Combine MSE reconstruction IP with QJL residual correction."""
        x_hat_mse = self.mse.dequantize(mse_out)
        ip_mse = x_hat_mse @ query  # (n,) for single query
        ip_qjl = self.qjl.estimate_ip(query, qjl_out)
        return ip_mse + ip_qjl  # unbiased!


print("ToyTurboQuant defined -- the full two-stage system.")

---
## 4. Verify unbiasedness

The moment of truth. We create vectors with known inner products and check whether the
two-stage estimate is unbiased. We compare against MSE-only to see the difference.

In [ ]:
section_header("Unbiasedness Verification", "Is E[estimate] = true value?")

torch.manual_seed(123)
dim = 64
n_trials = 200

# Fixed query
query = torch.randn(dim)
query = query / query.norm()

# Fixed database vector
x = torch.randn(dim)
x = x / x.norm()

true_ip = (query @ x).item()
print(f"True inner product: {true_ip:.6f}")

# Run many trials with different random seeds
mse_only_estimates = []
turboquant_estimates = []

for trial in range(n_trials):
    seed = trial * 17  # different random rotation each trial
    
    # MSE-only (3 bits)
    mse_q = ToyTurboQuantMSE(dim, bits=3, seed=seed)
    mse_out = mse_q.quantize(x.unsqueeze(0))
    x_hat = mse_q.dequantize(mse_out).squeeze(0)
    mse_only_estimates.append((query @ x_hat).item())
    
    # TurboQuant (3 bits = 2 MSE + 1 QJL)
    tq = ToyTurboQuant(dim, bit_width=3, seed=seed)
    mse_out, qjl_out = tq.quantize(x.unsqueeze(0))
    ip_est = tq.compute_inner_product(query, mse_out, qjl_out).item()
    turboquant_estimates.append(ip_est)

mse_only_estimates = torch.tensor(mse_only_estimates)
turboquant_estimates = torch.tensor(turboquant_estimates)

print(f"\nMSE-only (3 bits):")
print(f"  Mean estimate: {mse_only_estimates.mean().item():.6f}")
print(f"  Std:           {mse_only_estimates.std().item():.6f}")
print(f"  Bias:          {(mse_only_estimates.mean() - true_ip).item():.6f}")

print(f"\nTurboQuant (2+1 bits):")
print(f"  Mean estimate: {turboquant_estimates.mean().item():.6f}")
print(f"  Std:           {turboquant_estimates.std().item():.6f}")
print(f"  Bias:          {(turboquant_estimates.mean() - true_ip).item():.6f}")

In [ ]:
# Histogram: MSE-only vs TurboQuant estimates
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=mse_only_estimates.numpy(),
    name="MSE-only (3 bits)",
    opacity=0.6,
    marker_color="#e74c3c",
    nbinsx=30,
))

fig.add_trace(go.Histogram(
    x=turboquant_estimates.numpy(),
    name="TurboQuant (2+1 bits)",
    opacity=0.6,
    marker_color="#3498db",
    nbinsx=30,
))

# True value line
fig.add_vline(
    x=true_ip, line_width=3, line_dash="dash", line_color="green",
    annotation_text=f"True IP = {true_ip:.4f}",
    annotation_position="top right",
)

# Mean lines
fig.add_vline(
    x=mse_only_estimates.mean().item(),
    line_width=2, line_color="#e74c3c",
    annotation_text=f"MSE mean = {mse_only_estimates.mean().item():.4f}",
    annotation_position="bottom left",
)
fig.add_vline(
    x=turboquant_estimates.mean().item(),
    line_width=2, line_color="#3498db",
    annotation_text=f"TQ mean = {turboquant_estimates.mean().item():.4f}",
    annotation_position="bottom right",
)

fig.update_layout(
    title="Inner Product Estimates: MSE-Only (Biased) vs TurboQuant (Unbiased)",
    xaxis_title="Estimated inner product",
    yaxis_title="Count",
    barmode="overlay",
    template="plotly_white",
    height=450, width=800,
    legend=dict(x=0.02, y=0.98),
)

fig.show()

Key observations:

- The **MSE-only** distribution (red) is centered away from the true value -- it is biased.
- The **TurboQuant** distribution (blue) is centered on the true value -- it is unbiased.
- TurboQuant may have higher variance (wider spread), but the mean is correct.

For search and ranking tasks, unbiased estimates are critical: you want the *expected*
ranking to match the true ranking.

---
## 5. The bias-variance tradeoff

There is a fundamental tradeoff at play:

| Mode | Bias | Variance | Best for |
|------|------|----------|----------|
| MSE-only (all b bits to MSE) | Biased | Lower | Pure reconstruction |
| TurboQuant (b-1 + 1 split) | Unbiased | Slightly higher | Inner products, search, attention |

The MSE-only mode uses all bits for reconstruction, so its variance is lower. But the
bias means your estimates are systematically wrong.

The TurboQuant mode sacrifices 1 bit to QJL, increasing variance slightly, but eliminates
bias entirely. For inner product tasks, this is almost always the right choice.

This is exactly why the reference code has an `unbiased` flag: sometimes you want pure
reconstruction (set `unbiased=False`), and sometimes you want unbiased inner products
(set `unbiased=True`, the default).

In [ ]:
section_header("Bias-Variance Tradeoff", "Quantifying the difference")

# MSE-only
mse_bias = (mse_only_estimates.mean() - true_ip).abs().item()
mse_var = mse_only_estimates.var().item()
mse_mse_ip = ((mse_only_estimates - true_ip) ** 2).mean().item()  # MSE of the IP estimate

# TurboQuant
tq_bias = (turboquant_estimates.mean() - true_ip).abs().item()
tq_var = turboquant_estimates.var().item()
tq_mse_ip = ((turboquant_estimates - true_ip) ** 2).mean().item()

print(f"{'Metric':<30} {'MSE-only (3b)':<18} {'TurboQuant (2+1b)':<18}")
print("-" * 66)
print(f"{'|Bias|':<30} {mse_bias:<18.6f} {tq_bias:<18.6f}")
print(f"{'Variance':<30} {mse_var:<18.6f} {tq_var:<18.6f}")
print(f"{'MSE of IP estimate':<30} {mse_mse_ip:<18.6f} {tq_mse_ip:<18.6f}")
print(f"{'(= Bias^2 + Variance)':<30}")
print(f"\nRecall: MSE = Bias^2 + Variance.")
print(f"Even though TurboQuant has higher variance, its total MSE can be")
print(f"competitive because it eliminates the bias term entirely.")

---
## 6. Distortion comparison

At 3 bits total, the two modes allocate bits differently:

- **MSE-only**: all 3 bits go to MSE reconstruction
- **TurboQuant**: 2 bits MSE + 1 bit QJL

The MSE stage gets fewer bits in TurboQuant mode, so its reconstruction error is higher.
But the inner product distortion can be lower thanks to unbiasedness.

In [ ]:
section_header("Distortion Comparison", "MSE-only vs TurboQuant at 3 bits total")

torch.manual_seed(99)
dim = 64
n_vectors = 1000

X = torch.randn(n_vectors, dim)
X = X / X.norm(dim=-1, keepdim=True)

# MSE-only with 3 bits
mse3 = ToyTurboQuantMSE(dim, bits=3, seed=0)
mse3_out = mse3.quantize(X)
X_hat_mse3 = mse3.dequantize(mse3_out)
mse3_distortion = ((X - X_hat_mse3) ** 2).sum(dim=-1).mean().item()

# TurboQuant with 3 bits (2 MSE + 1 QJL)
tq3 = ToyTurboQuant(dim, bit_width=3, seed=0)
tq3_mse_out, tq3_qjl_out = tq3.quantize(X)
X_hat_tq3_mse = tq3.mse.dequantize(tq3_mse_out)  # MSE-only reconstruction
tq3_mse_distortion = ((X - X_hat_tq3_mse) ** 2).sum(dim=-1).mean().item()

# Inner product distortion: use 200 random queries
n_queries = 200
queries = torch.randn(n_queries, dim)
queries = queries / queries.norm(dim=-1, keepdim=True)

true_ips = queries @ X.T  # (n_queries, n_vectors)

# MSE-only IP estimates
est_ips_mse = queries @ X_hat_mse3.T
ip_distortion_mse = ((est_ips_mse - true_ips) ** 2).mean().item()

# TurboQuant IP estimates
est_ips_tq = torch.zeros_like(true_ips)
for i in range(n_queries):
    est_ips_tq[i] = tq3.compute_inner_product(queries[i], tq3_mse_out, tq3_qjl_out)
ip_distortion_tq = ((est_ips_tq - true_ips) ** 2).mean().item()

print(f"{'Metric':<35} {'MSE-only (3b)':<18} {'TurboQuant (2+1b)':<18}")
print("-" * 71)
print(f"{'MSE bits used':<35} {'3':<18} {'2':<18}")
print(f"{'QJL bits used':<35} {'0':<18} {'1':<18}")
print(f"{'Reconstruction MSE':<35} {mse3_distortion:<18.6f} {tq3_mse_distortion:<18.6f}")
print(f"{'Inner product MSE':<35} {ip_distortion_mse:<18.6f} {ip_distortion_tq:<18.6f}")
print(f"\nReconstruction MSE is HIGHER for TurboQuant (fewer MSE bits).")
print(f"But inner product MSE can be LOWER (because the estimate is unbiased).")

In [ ]:
# Bar chart comparison
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Reconstruction MSE (lower = better)",
                    "Inner Product MSE (lower = better)"],
)

methods = ["MSE-only (3b)", "TurboQuant (2+1b)"]
colors = ["#e74c3c", "#3498db"]

fig.add_trace(go.Bar(
    x=methods, y=[mse3_distortion, tq3_mse_distortion],
    marker_color=colors, name="Reconstruction MSE",
    text=[f"{mse3_distortion:.4f}", f"{tq3_mse_distortion:.4f}"],
    textposition="auto",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=methods, y=[ip_distortion_mse, ip_distortion_tq],
    marker_color=colors, name="IP MSE",
    text=[f"{ip_distortion_mse:.6f}", f"{ip_distortion_tq:.6f}"],
    textposition="auto",
), row=1, col=2)

fig.update_layout(
    title="The Tradeoff: Reconstruction vs. Inner Product Quality",
    template="plotly_white",
    height=400, width=800,
    showlegend=False,
)

fig.show()

---
## 7. Nearest neighbor search comparison

The real test: does unbiased inner product estimation translate to better search quality?
We measure recall@10 -- the fraction of true top-10 nearest neighbors that appear in the
estimated top-10.

In [ ]:
section_header("Nearest Neighbor Search", "500 vectors, 50 queries, k=10")

torch.manual_seed(777)
dim = 64
n_db = 500
n_queries = 50
k = 10

# Database vectors
db = torch.randn(n_db, dim)
db = db / db.norm(dim=-1, keepdim=True)

# Query vectors
queries = torch.randn(n_queries, dim)
queries = queries / queries.norm(dim=-1, keepdim=True)

# True top-k for each query
true_ips = queries @ db.T  # (n_queries, n_db)
true_topk = true_ips.topk(k, dim=1).indices  # (n_queries, k)

def recall_at_k(est_ips, true_topk, k):
    """Compute recall@k: fraction of true top-k found in estimated top-k."""
    est_topk = est_ips.topk(k, dim=1).indices
    recalls = []
    for i in range(len(est_ips)):
        true_set = set(true_topk[i].tolist())
        est_set = set(est_topk[i].tolist())
        recalls.append(len(true_set & est_set) / k)
    return sum(recalls) / len(recalls)

# MSE-only (3 bits)
mse_q = ToyTurboQuantMSE(dim, bits=3, seed=0)
mse_out = mse_q.quantize(db)
db_hat = mse_q.dequantize(mse_out)
est_ips_mse = queries @ db_hat.T
recall_mse = recall_at_k(est_ips_mse, true_topk, k)

# TurboQuant (3 bits = 2 MSE + 1 QJL)
tq = ToyTurboQuant(dim, bit_width=3, seed=0)
tq_mse_out, tq_qjl_out = tq.quantize(db)
est_ips_tq = torch.zeros(n_queries, n_db)
for i in range(n_queries):
    est_ips_tq[i] = tq.compute_inner_product(queries[i], tq_mse_out, tq_qjl_out)
recall_tq = recall_at_k(est_ips_tq, true_topk, k)

print(f"Recall@{k} Results:")
print(f"  MSE-only (3 bits):      {recall_mse:.4f}")
print(f"  TurboQuant (2+1 bits):  {recall_tq:.4f}")
print(f"\nDifference: {(recall_tq - recall_mse):.4f}")
if recall_tq >= recall_mse:
    print("TurboQuant wins -- unbiased IP estimates give better rankings.")
else:
    print("MSE-only wins on this run -- the toy quantizer may not show the")
    print("full benefit. The reference implementation with Lloyd-Max centroids")
    print("shows a clearer advantage for TurboQuant.")

In [ ]:
# Recall comparison bar chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=["MSE-only (3 bits)", "TurboQuant (2+1 bits)"],
    y=[recall_mse, recall_tq],
    marker_color=["#e74c3c", "#3498db"],
    text=[f"{recall_mse:.3f}", f"{recall_tq:.3f}"],
    textposition="auto",
    width=0.5,
))

fig.update_layout(
    title=f"Recall@{k}: MSE-Only vs TurboQuant (3 bits total)",
    yaxis_title=f"Recall@{k}",
    yaxis_range=[0, 1],
    template="plotly_white",
    height=400, width=500,
)

fig.show()

---
## 8. The reference implementation

Let's look at the actual TurboQuant class from the reference code
(`reference-code/turboquant-torch/turboquant/core.py`). This is what a production
implementation looks like.

In [ ]:
section_header("Reference Implementation", "turboquant/core.py")

# The actual TurboQuant class (from the reference code)
reference_code = """
class TurboQuant:
    def __init__(self, dim, bit_width=3, unbiased=True, seed=0):
        self.dim = dim
        self.bit_width = bit_width
        self.unbiased = unbiased
        if unbiased:
            if bit_width < 2:
                raise ValueError("Unbiased mode requires bit_width >= 2")
            mse_bits = bit_width - 1
        else:
            mse_bits = bit_width
        self.mse = TurboQuantMSE(dim, bits=mse_bits, seed=seed)
        self.qjl = QJL(dim, proj_dim=dim, seed=seed + 1) if unbiased else None

    def quantize(self, x):
        mse_out = self.mse.quantize(x)
        qjl_output = None
        if self.qjl is not None:
            residual = x - self.mse.dequantize(mse_out)
            qjl_output = self.qjl.quantize(residual)
        return TurboQuantOutput(mse_codes=mse_out.codes, mse_norms=mse_out.norms,
                                qjl_output=qjl_output, bit_width=self.bit_width,
                                dim=self.dim)

    def compute_inner_product(self, query, output):
        mse_out = MSEQuantizedOutput(codes=output.mse_codes, norms=output.mse_norms,
                                     bit_width=self.bit_width - (1 if self.unbiased else 0))
        x_hat = self.mse.dequantize(mse_out)
        ip = query @ x_hat.t() if query.dim() > 1 else x_hat @ query
        if self.qjl is not None and output.qjl_output is not None:
            qjl_ip = self.qjl.estimate_inner_product(query, output.qjl_output)
            ip = ip + (qjl_ip.squeeze(0) if query.dim() == 1 else qjl_ip)
        return ip
"""
print(reference_code)

### Anatomy of the reference code

**The `unbiased` flag** controls everything:

- `unbiased=True` (default): splits the bit budget as (b-1) MSE + 1 QJL. Creates both
  a `TurboQuantMSE` and a `QJL` object. Inner products are unbiased.
- `unbiased=False`: gives all b bits to MSE. No QJL object is created. Inner products
  are biased but reconstruction MSE is lower.

**The bit budget split** happens in `__init__`:
```python
mse_bits = bit_width - 1  # if unbiased
mse_bits = bit_width      # if not unbiased
```

**Residual computation** happens in `quantize()`:
```python
residual = x - self.mse.dequantize(mse_out)
qjl_output = self.qjl.quantize(residual)
```
The MSE quantizer runs first. Then we compute the error it made (the residual) and
hand that to QJL. This is sequential: QJL sees the residual, not the original vector.

**Asymmetric IP estimation** in `compute_inner_product()`:
```python
ip = query @ x_hat.t()  # MSE component (query is full precision)
qjl_ip = self.qjl.estimate_inner_product(query, output.qjl_output)  # QJL correction
ip = ip + qjl_ip  # combine
```
The query is never quantized -- it stays in full precision. Only the database vectors
are quantized. This asymmetric setup is standard in vector search: you quantize the
database once, then run many full-precision queries against it.

---
## Exercise: Compression ratio

For 3-bit TurboQuant (2 bits MSE + 1 bit QJL), estimate the compression ratio vs float32.

**Setup**: Each coordinate uses 3 bits total. We also need to store one float32 norm per
vector (for the MSE stage). For QJL, the residual norms are also stored (another float32
per vector).

For a vector of dimension d = 128:
- Original: d * 32 bits = ?
- Compressed: d * 3 bits (codes) + 32 bits (MSE norm) + 32 bits (QJL residual norm) = ?
- Compression ratio = original / compressed = ?

In [ ]:
section_header("Exercise: Compression Ratio", "3-bit TurboQuant vs float32")

d = 128
bits_per_coord = 3

# Fill in the blanks
original_bits = d * 32
code_bits = d * bits_per_coord
norm_bits = 32  # MSE norm (1 float32)
# Note: in the simplified analysis we count just the MSE norm.
# The QJL residual norm adds another 32 bits.
compressed_bits = code_bits + norm_bits

ratio = original_bits / compressed_bits

print(f"Dimension: {d}")
print(f"Original:   {d} coords x 32 bits = {original_bits} bits")
print(f"Compressed: {d} coords x {bits_per_coord} bits + 32 bits (norm) = {compressed_bits} bits")
print(f"Compression ratio: {original_bits} / {compressed_bits} = {ratio:.1f}x")
print()

check_intuition(
    ratio, 9.8,
    hint="Each coordinate: 3 bits vs 32 bits. Plus 32 bits for the norm.",
    tolerance=0.1,
    label="Compression ratio",
)

print(f"\nFor a database of 1M vectors at d={d}:")
original_mb = (1_000_000 * original_bits) / 8 / 1e6
compressed_mb = (1_000_000 * compressed_bits) / 8 / 1e6
print(f"  Original:   {original_mb:.1f} MB")
print(f"  Compressed: {compressed_mb:.1f} MB")
print(f"  Savings:    {original_mb - compressed_mb:.1f} MB")

---
## Summary

| Concept | Key takeaway |
|---|---|
| MSE-only bias | Scalar quantization shrinks inner product estimates systematically |
| (b-1) + 1 split | Give bulk of bits to MSE, reserve 1 bit for QJL on the residual |
| Residual is small | QJL operates on the small quantization error, keeping variance low |
| Unbiased IP | E[MSE component + QJL correction] = true inner product |
| Bias-variance tradeoff | Unbiased mode trades slightly higher variance for zero bias |
| `unbiased` flag | Reference code supports both modes depending on the use case |
| Compression | 3-bit TurboQuant achieves ~9.8x compression at d=128 |

**Next module**: How close is TurboQuant to the information-theoretic limit? Shannon bounds
tell us how good *any* quantizer could possibly be.